In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.shipments
(
    shipment_id INT,
    order_id INT,

    carrier STRING,
    tracking_number STRING,

    shipment_status STRING,

    shipped_date TIMESTAMP,
    delivered_date TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("de_learning_workspace.bronze.shipments")

silver_df = (
    bronze_df

    .dropDuplicates(["shipment_id"])

    .withColumn(
        "carrier",
        coalesce(trim(col("carrier")), lit("Unknown"))
    )

    .withColumn(
        "tracking_number",
        coalesce(trim(col("tracking_number")), lit("Not Assigned"))
    )

    .withColumn(
        "shipment_status",
        initcap(trim(col("shipment_status")))
    )

    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("de_learning_workspace.silver.shipments")
)